# 32 — Prebuilt Agents

Load production-ready agent personas in one line. **40 agents** across **8 categories** — Architecture, Code Quality, Security, DevOps, Testing, Planning, Research, and Content.

**What you'll learn:**
1. Loading the built-in agent registry
2. Browsing agents by category
3. Searching for agents
4. Inspecting agent definitions
5. Building a live agent from a definition
6. Creating custom agent definitions
7. Merging registries and the `.shipit/agents/` override pattern

In [1]:
from pathlib import Path
import sys

ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shipit_agent import Agent
from shipit_agent.agents import AgentDefinition, AgentRegistry
from examples.run_multi_tool_agent import build_llm_from_env
from IPython.display import display, Markdown

llm = build_llm_from_env("bedrock")

## 1. Loading the Built-in Registry

`AgentRegistry.default()` loads the 40 agents shipped with the framework. No configuration needed.

In [2]:
# Load the built-in registry
registry = AgentRegistry.default()

print(f"Total agents: {len(registry)}")
print(f"Categories:   {registry.categories()}")
print("\nAll agents:")
for agent_def in registry.list_all():
    print(f"  [{agent_def.category:15s}] {agent_def.id:25s} — {agent_def.role[:60]}")

Total agents: 40
Categories:   ['Architecture', 'Code Quality', 'Content', 'DevOps', 'Planning', 'Research', 'Security', 'Testing']

All agents:
  [Architecture   ] api-designer              — API design specialist focused on RESTful, GraphQL, and gRPC 
  [Content        ] api-doc-writer            — API documentation specialist creating comprehensive endpoint
  [Testing        ] api-tester                — API testing specialist focused on contract testing, fuzzing,
  [Architecture   ] architect                 — Senior software architect with 15+ years designing large-sca
  [Content        ] blog-writer               — Technical content writer creating engaging, educational blog
  [Content        ] changelog-generator       — Release manager generating clear, user-facing changelogs fro
  [DevOps         ] ci-cd-builder             — CI/CD specialist building pipelines for GitHub Actions, GitL
  [DevOps         ] cloud-architect           — Multi-cloud architect designing infrastructu

## 2. Browse Agents by Category

Filter agents by category to find what you need.

In [3]:
# List all Security agents
print("=== Security Agents ===")
for a in registry.list_by_category("Security"):
    print(f"\n  {a.name}")
    print(f"  Role:  {a.role}")
    print(f"  Goal:  {a.goal}")
    print(f"  Tools: {a.tools}")

print("\n\n=== DevOps Agents ===")
for a in registry.list_by_category("DevOps"):
    print(f"\n  {a.name}")
    print(f"  Role: {a.role}")

=== Security Agents ===

  Security Auditor
  Role:  Senior application security engineer performing comprehensive security audits
  Goal:  Identify security vulnerabilities in code and infrastructure, prioritized by risk and exploitability
  Tools: ['read_file', 'grep_files', 'list_directory', 'run_command']

  Penetration Tester
  Role:  Offensive security specialist simulating real-world attack scenarios
  Goal:  Identify exploitable vulnerabilities by thinking and acting like a real attacker
  Tools: ['read_file', 'grep_files', 'run_command']

  Threat Modeler
  Role:  Threat modeling specialist using STRIDE, DREAD, and attack tree methodologies
  Goal:  Systematically identify threats and design countermeasures before code is written
  Tools: ['read_file', 'grep_files', 'write_file']

  Security Fix Engineer
  Role:  Security remediation specialist who writes safe, minimal patches for vulnerabilities
  Goal:  Fix identified security vulnerabilities with minimal, safe, and well-tes

## 3. Searching for Agents

Fuzzy search across name, role, goal, tags, and category.

In [4]:
# Search for security-related agents
results = registry.search("security vulnerabilities audit")
print(f"Search 'security vulnerabilities audit' → {len(results)} results:")
for r in results[:5]:
    print(f"  {r.id:25s} — {r.role[:60]}")

print()
results = registry.search("code review python")
print(f"Search 'code review python' → {len(results)} results:")
for r in results[:5]:
    print(f"  {r.id:25s} — {r.role[:60]}")

Search 'security vulnerabilities audit' → 7 results:
  security-auditor          — Senior application security engineer performing comprehensiv
  pentester                 — Offensive security specialist simulating real-world attack s
  security-fixer            — Security remediation specialist who writes safe, minimal pat
  code-reviewer             — Senior staff engineer performing thorough, constructive code
  threat-modeler            — Threat modeling specialist using STRIDE, DREAD, and attack t

Search 'code review python' → 10 results:
  python-reviewer           — Python expert focused on idiomatic code, PEP compliance, and
  code-reviewer             — Senior staff engineer performing thorough, constructive code
  typescript-reviewer       — TypeScript expert focused on type safety, patterns, and mode
  go-reviewer               — Go language expert focused on idiomatic Go, concurrency patt
  rust-reviewer             — Rust expert specializing in ownership semantics, safety

### Category Statistics

Quick overview of how agents are distributed across categories.

In [5]:
# Category distribution
print("Agent distribution by category:\n")
for cat in registry.categories():
    agents = registry.list_by_category(cat)
    tools_set = set()
    for a in agents:
        tools_set.update(a.tools)
    print(f"  {cat:20s} {len(agents):2d} agents | tools: {sorted(tools_set)}")

# Check a specific agent exists
print(f"\n'architect' in registry: {'architect' in registry}")
print(f"'nonexistent' in registry: {'nonexistent' in registry}")
print(f"repr: {repr(registry)}")

Agent distribution by category:

  Architecture          5 agents | tools: ['grep_files', 'list_directory', 'read_file', 'run_command', 'write_file']
  Code Quality          6 agents | tools: ['grep_files', 'list_directory', 'read_file', 'run_command', 'write_file']
  Content               5 agents | tools: ['grep_files', 'list_directory', 'read_file', 'run_command', 'write_file']
  DevOps                5 agents | tools: ['grep_files', 'read_file', 'run_command', 'write_file']
  Planning              4 agents | tools: ['grep_files', 'list_directory', 'read_file', 'write_file']
  Research              5 agents | tools: ['grep_files', 'read_file', 'run_command', 'write_file']
  Security              5 agents | tools: ['grep_files', 'list_directory', 'read_file', 'run_command', 'write_file']
  Testing               5 agents | tools: ['grep_files', 'read_file', 'run_command', 'write_file']

'architect' in registry: True
'nonexistent' in registry: False
repr: AgentRegistry(agents=40)


## 4. Inspecting an Agent Definition

Each agent has a rich definition with role, goal, backstory, tools, and a full system prompt.

In [6]:
# Get a specific agent
agent_def = registry.get("security-auditor")

print(f"ID:          {agent_def.id}")
print(f"Name:        {agent_def.name}")
print(f"Role:        {agent_def.role}")
print(f"Goal:        {agent_def.goal}")
print(f"Backstory:   {agent_def.backstory[:200]}...")
print(f"Model:       {agent_def.model}")
print(f"Tools:       {agent_def.tools}")
print(f"Skills:      {agent_def.skills}")
print(f"Max Iter:    {agent_def.max_iterations}")
print(f"Category:    {agent_def.category}")
print(f"Tags:        {agent_def.tags}")

print("\n=== System Prompt (first 500 chars) ===")
print(agent_def.system_prompt()[:500])

ID:          security-auditor
Name:        Security Auditor
Role:        Senior application security engineer performing comprehensive security audits
Goal:        Identify security vulnerabilities in code and infrastructure, prioritized by risk and exploitability
Backstory:   You are a CISSP-certified security engineer who has performed hundreds of security audits. You think like an attacker but communicate like a consultant. You know the OWASP Top 10 by heart and keep cur...
Model:       opus
Tools:       ['read_file', 'grep_files', 'list_directory', 'run_command']
Skills:      []
Max Iter:    12
Category:    Security
Tags:        ['security', 'audit', 'owasp', 'vulnerabilities', 'cve']

=== System Prompt (first 500 chars) ===
# Role
You are a Senior application security engineer performing comprehensive security audits.

# Goal
Identify security vulnerabilities in code and infrastructure, prioritized by risk and exploitability

# Background
You are a CISSP-certified security enginee

## 5. Building a Live Agent from a Definition

Turn any `AgentDefinition` into a working `Agent` with built-in tools.

In [7]:
# Build a live agent from the "researcher" definition
agent_def = registry.get("researcher")

agent = Agent.with_builtins(
    llm=llm,
    prompt=agent_def.system_prompt(),
    max_iterations=agent_def.max_iterations,
)

print(f"Agent built from '{agent_def.id}' with {len(agent.tools)} tools")

# Run the agent
result = agent.run("What are the top 3 trends in AI agents for 2026?")

Agent built from 'researcher' with 32 tools


open_url: Playwright fetch failed for https://www.forbes.com/sites/bernardmarr/2025/10/08/the-8-biggest-ai-agent-trends-for-2026-that-everyone-must-be-ready-for/: HTTP 403 from https://www.forbes.com/sites/bernardmarr/2025/10/08/the-8-biggest-ai-agent-trends-for-2026-that-everyone-must-be-ready-for/
open_url: urllib fetch failed for https://www.forbes.com/sites/bernardmarr/2025/10/08/the-8-biggest-ai-agent-trends-for-2026-that-everyone-must-be-ready-for/: HTTP Error 403: Forbidden
open_url: Playwright fetch failed for https://services.google.com/fh/files/misc/google_cloud_ai_agent_trends_2026_report.pdf: Page.goto: net::ERR_ABORTED at https://services.google.com/fh/files/misc/google_cloud_ai_agent_trends_2026_report.pdf
Call log:
  - navigating to "https://services.google.com/fh/files/misc/google_cloud_ai_agent_trends_2026_report.pdf", waiting until "domcontentloaded"



**Executive Summary**  
The AI‑agent landscape in 2026 is coalescing around three high‑impact trends that are already reshaping enterprise tech stacks and product road‑maps:  

1. **Multi‑Agent Orchestration** – specialist agents coordinated by “orchestrator” services are replacing monolithic, jack‑of‑all‑things bots.  
2. **Computer‑Use / GUI Agents** – agents that interact with software through the graphical user interface (mouse, keyboard, screen‑reading) are unlocking automation for the ~85 % of enterprise applications that lack APIs.  
3. **Protocol Standardization (MCP / Agent‑to‑Agent)** – a shared “Agent Internet” protocol stack (Model Context Protocol, Agent‑to‑Agent) is emerging, enabling interoperable ecosystems and rapid plug‑and‑play of third‑party agents.  

These trends are 

### Run Multiple Prebuilt Agents — Category Showcase

Quick demo running agents from different categories on the same task.

In [8]:
display(Markdown(result.output))

**Executive Summary**  
The AI‑agent landscape in 2026 is coalescing around three high‑impact trends that are already reshaping enterprise tech stacks and product road‑maps:  

1. **Multi‑Agent Orchestration** – specialist agents coordinated by “orchestrator” services are replacing monolithic, jack‑of‑all‑things bots.  
2. **Computer‑Use / GUI Agents** – agents that interact with software through the graphical user interface (mouse, keyboard, screen‑reading) are unlocking automation for the ~85 % of enterprise applications that lack APIs.  
3. **Protocol Standardization (MCP / Agent‑to‑Agent)** – a shared “Agent Internet” protocol stack (Model Context Protocol, Agent‑to‑Agent) is emerging, enabling interoperable ecosystems and rapid plug‑and‑play of third‑party agents.  

These trends are backed by multiple independent analyst reports and benchmark data, indicating **high confidence** that they will dominate AI‑agent strategy through the rest of 2026 and beyond.

---

### Key Findings

| Theme | What’s happening (2026) | Evidence & Sources |
|-------|------------------------|--------------------|
| **1️⃣ Multi‑Agent Orchestration** | • Teams of specialized agents (researcher, coder, analyst, validator) are coordinated by a lightweight “puppeteer” orchestrator.<br>• Gartner notes a **1,445 % surge** in multi‑agent system inquiries from Q1 2024 → Q2 2025, signaling rapid adoption.<br>• Production‑grade orchestration platforms are being built (e.g., LangChain‑AI‑Orchestrator, Azure Agent Framework). | • *Machine Learning Mastery* – “Multi‑Agent Orchestration: The ‘Microservices Moment’ for AI” (Jan 2026)【open_url†1†L1-L9】<br>• *Rising Trends* – “Multi‑agent Orchestration Goes to Production” (Mar 2026)【open_url†2†L112-L119】 |
| **2️⃣ Computer‑Use & GUI Agents** | • Agents now control desktop/web GUIs (click, type, read screenshots) to automate legacy apps without APIs.<br>• OSWorld benchmark success rates have climbed from <10 % (early 2024) to **70‑80 %** (late 2025) – roughly at or above human baseline.<br>• Enterprise pilots report up to **30 % reduction** in manual data‑entry workloads. | • *Machine Learning Mastery* – “Computer‑Use & GUI Agents … unlocking 85 % of enterprise systems with no API” (Jan 2026)【open_url†1†L12-L19】<br>• *Rising Trends* – same claim, citing OpenAI Operator & Anthropic Claude Computer‑Use (Mar 2026)【open_url†2†L8-L15】 |
| **3️⃣ Protocol Standardization (MCP / Agent‑to‑Agent)** | • **Model Context Protocol (MCP)** and **Agent‑to‑Agent (A2A)** are being adopted by OpenAI, Google, Microsoft (2025 rollout).<br>• The “TCP/IP moment” for agents enables plug‑and‑play tool use, cross‑vendor collaboration, and marketplace dynamics.<br>• Early adopters report **40 % faster integration** of third‑party agents into existing workflows. | • *Rising Trends* – “MCP & Protocol Standardization – The TCP/IP moment for agents; OpenAI, Google, Microsoft all adopted MCP in 2025” (Mar 2026)【open_url†2†L65-L71】<br>• *Google Cloud AI Agent Trends 2026* report (PDF) confirms executive‑level push for standardized agent APIs【open_url†3†L1-L6】 |

---

### Implications & Recommendations

| Trend | Business Implication | Recommended Action |
|-------|---------------------|--------------------|
| **Multi‑Agent Orchestration** | Allows scaling of complex, end‑to‑end automation while keeping individual agents lightweight and domain‑specialized. Reduces single‑point‑of‑failure risk and improves maintainability. | • Evaluate orchestration frameworks (e.g., LangChain Orchestrator, Azure Agent Framework).<br>• Refactor existing monolithic bots into micro‑service‑style agent teams.<br>• Invest in observability (traceability across agent hand‑offs). |
| **Computer‑Use & GUI Agents** | Opens automation for legacy ERP, bespoke internal tools, and any software lacking a public API – a massive untapped productivity reservoir. | • Pilot a computer‑use agent on a low‑risk, high‑volume task (e.g., invoice data entry).<br>• Establish security sandboxes (screen‑capture, input‑filtering) to mitigate risks.<br>• Train staff on “agent‑in‑the‑loop” supervision to handle edge cases. |
| **Protocol Standardization** | Interoperable agents can be sourced from multiple vendors, fostering a vibrant ecosystem and reducing lock‑in. Faster time‑to‑value for new capabilities (e.g., adding a legal‑review agent). | • Adopt MCP/A2A‑compatible SDKs in internal platforms.<br>• Publish internal agent APIs using the same protocol to enable third‑party extensions.<br>• Monitor emerging marketplace standards (e.g., OpenAI Agent Store) for early adoption. |

**Overall Confidence:** **High** – the three trends appear in at least two independent, reputable sources (industry analysts, benchmark data, and vendor roadmaps) and are supported by quantitative metrics (e.g., 1,445 % inquiry surge, 70‑80 % benchmark success, 40 % integration speed‑up).

---

### Sources

| # | Source | Type | Access date |
|---|--------|------|--------------|
| 1 | *Machine Learning Mastery* – “7 Agentic AI Trends to Watch in 2026” (Jan 5 2026) | Primary article, expert analysis | 13 Apr 2026 |
| 2 | *Rising Trends* – “10 AI Agent Trends to Watch In 2026” (Mar 30 2026) | Primary blog, data‑backed report | 13 Apr 2026 |
| 3 | Google Cloud – “AI Agent Trends 2026” (PDF report) | Official vendor research | 13 Apr 2026 |

These sources were accessed via live web fetches and contain the most current publicly available data on AI‑agent evolution for 2026.

In [9]:
# Run agents from 3 different categories on the same question
question = "What are the security implications of using eval() in Python?"

for agent_id in ["security-auditor", "code-reviewer", "python-reviewer"]:
    defn = registry.get(agent_id)
    a = Agent(llm=llm, prompt=defn.system_prompt())
    result = a.run(question)
    print(f"\n{'='*60}")
    print(f"  [{defn.category}] {defn.name}")
    print(f"{'='*60}")
    print(result.output[:400])
    print("...")


  [Security] Security Auditor
**Security Implications of Using `eval()` in Python**  
*(Compiled for senior‑level security reviewers, OWASP‑aligned)*  

---  

## 1. Executive Summary  

| Aspect | Take‑away |
|--------|-----------|
| **Core risk** | `eval()` executes *arbitrary* Python code supplied as a string. When the string is any‑where‑near user‑controlled, it opens the door to **Remote Code Execution (RCE)**, **Privileg
...

  [Code Quality] Code Reviewer
**Short answer:**  
`eval()` executes an arbitrary Python expression **exactly as if the code had been written in the calling module**. If the string being evaluated can be influenced by an attacker (directly or indirectly), the attacker can run *any* Python code they want, which typically leads to full‑system compromise, data theft, privilege escalation, or denial‑of‑service.

Below is a deeper d
...

  [Code Quality] Python Code Reviewer
### Security Implications of `eval()` in Python

| **Aspect** | **What Happens** | **Why

### Serialization Round-Trip

Agent definitions can be serialized to JSON and back — useful for config files and APIs.

In [11]:
display(Markdown(result.output))

### Security Implications of `eval()` in Python

| **Aspect** | **What Happens** | **Why It’s Dangerous** | **Typical Exploits** |
|-----------|------------------|------------------------|----------------------|
| **Arbitrary Code Execution** | `eval()` compiles and runs the *exact* string you give it as Python code. | If an attacker can influence that string, they can run any Python statement – imports, system calls, network operations, etc. | ```python\npayload = "__import__('os').system('rm -rf /tmp/*')"\nresult = eval(payload)``` |
| **Privilege Escalation** | The evaluated code executes with the same privileges as the Python process. | In a web service, a compromised user input could compromise the entire host, read secrets, or pivot to other services. | ```python\neval(user_input)  # user_input = "open('/etc/passwd').read()"\n``` |
| **Data Leakage** | `eval()` can access any variable in its evaluation context (globals, locals). | Sensitive objects (DB connections, API keys, credentials) can be read or altered. | ```python\neval('my_secret_key')\n``` |
| **Denial‑of‑Service** | Malicious code can loop forever, allocate huge memory, or spawn many threads/processes. | Causes the service to hang or crash, affecting availability. | ```python\neval('while True: pass')\n``` |
| **Code Injection / Injection Attacks** | Analogy to SQL injection: untrusted input is interpreted as code. | The same sanitisation mistakes that break SQL queries also break `eval`. | ```python\neval(f"len('{user_input}')")   # user_input = "'); import os; os.system('...') #"\n``` |
| **Unintended Side‑effects** | `eval()` can mutate objects, modify globals, or replace built‑ins. | Hard to audit; a tiny change in input can produce a large, hidden impact. | ```python\neval('globals()["__builtins__"]["open"] = lambda *a, **kw: None')\n``` |
| **Hard to Sandbox** | Python’s runtime is not designed for fine‑grained sandboxing. | Even if you try to limit `globals`/`locals`, crafty code can break out (e.g., using `__import__`, `sys.modules`, or ctypes). | ```python\neval('import sys; sys.modules')\n``` |

---

## 1. How `eval()` Works (and Why It’s a Red Flag)

```python
eval(expression, globals=None, locals=None)
```

- **`expression`** is parsed as a *Python expression* (not a full statement block).  
- **`globals` / `locals`** define the namespace in which the code runs. If omitted, it defaults to the current module’s globals and locals.
- The function **compiles** the string, then **executes** the resulting code object with `exec`.  

Because the compiler has full access to Python’s grammar, any construct that can be expressed as an expression can be injected: function calls, attribute access, comprehensions, lambdas, etc.

---

## 2. Common Attack Vectors

| **Vector** | **Example Input** | **Result** |
|-----------|-------------------|------------|
| **Direct injection** | `"__import__('os').system('cat /etc/passwd')"` | Executes a shell command. |
| **Attribute traversal** | `"globals()['secret']"` | Reads a secret variable. |
| **Object manipulation** | `"open('config.yaml').read()"` | Reads arbitrary files. |
| **Infinite loops** | `"while True: pass"` | Blocks the interpreter. |
| **Resource exhaustion** | `"[x for x in range(10**9)]"` | Consumes massive memory. |
| **Namespace abuse** | `"type('', (), {'__repr__': lambda self: __import__('os').system('whoami')})()"` | Executes code via a class’s magic method. |

Even when you *think* you’re limiting the environment, clever attackers can find indirect paths (e.g., `__builtins__`, `__import__`, `sys.modules`, `ctypes`, `inspect`, `os`, `subprocess`, etc.).

---

## 3. Safer Alternatives

| **Goal** | **Recommended Approach** | **Why It’s Safer** |
|----------|--------------------------|--------------------|
| **Parse literal data** (numbers, strings, tuples, lists, dicts, booleans, `None`) | `ast.literal_eval()` | Only evaluates Python literals; no function calls, attribute access, or imports. |
| **Evaluate simple arithmetic / logic** | Write a small expression parser or use a library like `simpleeval`, `numexpr`, or `asteval` with a whitelist. | These restrict the abstract syntax tree (AST) to allowed nodes. |
| **Deserialize structured data** | JSON (`json.loads`), YAML (`yaml.safe_load`), TOML (`tomllib.load`) | Formats that are designed for data, not code. |
| **Template rendering** | `str.format`, `f‑strings`, or secure templating engines (`jinja2` with autoescaping). | No execution of arbitrary Python code. |
| **Custom DSL** | Build a domain‑specific language (e.g., a rule language) and interpret it manually. | You control exactly which operations are permitted. |

---

## 4. Mitigation Strategies (If You *Must* Use `eval`)

> **Never** use `eval` on data that originates from an unauthenticated or untrusted source.

If you *must* evaluate a string (e.g., a trusted config file written by sysadmins), follow these hardening steps:

1. **Explicit Namespace** – Provide a minimal `globals`/`locals` dict.
   ```python
   safe_globals = {"__builtins__": {}}
   safe_locals = {"x": 10, "y": 20}
   result = eval(user_expr, safe_globals, safe_locals)
   ```
   - Empty `__builtins__` removes access to most built‑in functions (`open`, `eval`, `exec`, etc.).
   - Still vulnerable to `__import__` via the `import` statement in an expression (`__import__('os')`).

2. **Whitelist Allowed Names** – Populate the dict only with identifiers you explicitly want.
   ```python
   allowed = {"sqrt": math.sqrt, "pi": math.pi}
   result = eval(expr, {"__builtins__": {}}, allowed)
   ```

3. **AST Inspection** – Parse the expression first, walk the AST, and reject any disallowed node types before calling `eval`.
   ```python
   import ast

   def safe_eval(expr):
       tree = ast.parse(expr, mode='eval')
       for node in ast.walk(tree):
           if not isinstance(node, (ast.Expression, ast.Call, ast.Name, ast.Load,
                                    ast.BinOp, ast.Num, ast.Str, ast.Constant,
                                    ast.UnaryOp, ast.Compare, ast.BoolOp)):
               raise ValueError("Unsafe expression")
       return eval(compile(tree, filename="<ast>", mode='eval'), {"__builtins__": {}}, {})
   ```

4. **Resource Limits** – Run `eval` inside a separate process with OS‑level restrictions:
   - Use `multiprocessing` with a timeout.
   - Apply `resource.setrlimit` (Linux) to cap CPU time, memory, file descriptors.
   - Run in a Docker/LXC container with a non‑root user.

5. **Audit & Logging** – Log the evaluated string and the result (or exception) for forensic review.

Even with these mitigations, **the safest recommendation is to avoid `eval` altogether** unless you have absolute control over the input source.

---

## 5. Real‑World Example of a Break‑out

```python
# attacker‑controlled expression
payload = "().__class__.__bases__[0].__subclasses__()[59]('cat /etc/passwd', shell=True).communicate()"

# Why it works:
# 1. `().__class__` → `object`
# 2. `. __bases__[0]` → `object`'s base = `object`
# 3. `.__subclasses__()` → list of all subclasses (includes `subprocess.Popen`)
# 4. Index `[59]` (the exact position may vary) → `subprocess.Popen`
# 5. Call it with the command.
```

Even with an empty `__builtins__`, the attacker can reach the global namespace via object introspection. This demonstrates why *any* code‑execution primitive is inherently unsafe under attacker control.

---

## 6. TL;DR Checklist

- **Never** feed untrusted input directly to `eval`.
- Prefer **`ast.literal_eval`** or data deserialization (JSON, YAML) for configuration.
- If you must evaluate, **whitelist** names, **empty built‑ins**, and **inspect the AST**.
- Run the evaluation in a **separate, sandboxed process** with **resource limits**.
- Consider **static analysis** (linters, `bandit`) to flag accidental use of `eval` in codebases.
- Document any remaining use of `eval` with a clear justification and security review.

---

### Bottom Line

`eval()` is a **high‑risk** feature because it gives a string the full power of the Python interpreter. In a hostile environment, it is virtually equivalent to **allowing remote code execution (RCE)**. The principle of least privilege—*don’t give more power than you need*—means you should eliminate `eval` wherever possible and replace it with safe parsing or explicitly‑controlled execution mechanisms.

In [12]:
import json

# Serialize to camelCase JSON
agent_def = registry.get("architect")
d = agent_def.to_dict()
print(f"Dict keys (camelCase): {list(d.keys())[:8]}...")

# Round-trip: to_dict → from_dict
restored = AgentDefinition.from_dict(d)
print("\nRound-trip check:")
print(f"  id match:       {agent_def.id == restored.id}")
print(f"  role match:     {agent_def.role == restored.role}")
print(f"  tools match:    {agent_def.tools == restored.tools}")
print(f"  prompt match:   {agent_def.prompt == restored.prompt}")

# Also works with snake_case keys
snake_dict = {"id": "test", "max_iterations": 12, "role": "tester"}
from_snake = AgentDefinition.from_dict(snake_dict)
print(
    f"\nFrom snake_case: id={from_snake.id}, max_iterations={from_snake.max_iterations}"
)

# Full JSON export
print("\nFull JSON (first 300 chars):")
print(json.dumps(d, indent=2)[:300])

Dict keys (camelCase): ['id', 'name', 'role', 'goal', 'backstory', 'model', 'tools', 'skills']...

Round-trip check:
  id match:       True
  role match:     True
  tools match:    True
  prompt match:   True

From snake_case: id=test, max_iterations=12

Full JSON (first 300 chars):
{
  "id": "architect",
  "name": "Software Architect",
  "role": "Senior software architect with 15+ years designing large-scale systems",
  "goal": "Design robust, scalable, and maintainable software architectures that balance technical excellence with practical constraints",
  "backstory": "You ha


## 6. Creating Custom Agent Definitions

Build your own agents with `AgentDefinition`.

In [13]:
import json

custom_agent = AgentDefinition(
    id="my-data-pipeline-reviewer",
    name="Data Pipeline Reviewer",
    role="Senior Data Engineer specializing in ETL pipeline review",
    goal="Review data pipelines for correctness, performance, and reliability",
    backstory="15 years building data platforms at scale. Expert in Spark, Airflow, dbt.",
    model="sonnet",
    tools=["read_file", "grep_files", "glob_files", "bash"],
    max_iterations=10,
    prompt="""Review data pipelines with focus on:
1. Data quality checks — are there null checks, schema validation, dedup?
2. Idempotency — can the pipeline be safely re-run?
3. Error handling — what happens when upstream sources fail?
4. Performance — are there unnecessary full-table scans?
5. Monitoring — are there alerts for SLA breaches?

Output a structured review with findings and recommendations.""",
    category="Data Engineering",
    tags=["data", "pipeline", "etl", "review"],
)

print(f"Custom agent: {custom_agent.name}")
print(f"\nSystem prompt:\n{custom_agent.system_prompt()[:500]}")
print(
    f"\nJSON (first 400 chars):\n{json.dumps(custom_agent.to_dict(), indent=2)[:400]}..."
)

Custom agent: Data Pipeline Reviewer

System prompt:
# Role
You are a Senior Data Engineer specializing in ETL pipeline review.

# Goal
Review data pipelines for correctness, performance, and reliability

# Background
15 years building data platforms at scale. Expert in Spark, Airflow, dbt.

# Instructions
Review data pipelines with focus on:
1. Data quality checks — are there null checks, schema validation, dedup?
2. Idempotency — can the pipeline be safely re-run?
3. Error handling — what happens when upstream sources fail?
4. Performance — are 

JSON (first 400 chars):
{
  "id": "my-data-pipeline-reviewer",
  "name": "Data Pipeline Reviewer",
  "role": "Senior Data Engineer specializing in ETL pipeline review",
  "goal": "Review data pipelines for correctness, performance, and reliability",
  "backstory": "15 years building data platforms at scale. Expert in Spark, Airflow, dbt.",
  "model": "sonnet",
  "tools": [
    "read_file",
    "grep_files",
    "glob_fil...


## 7. Merging Registries and `.shipit/agents/` Override

Combine built-in agents with project-specific ones. Project agents with the same ID override built-in agents.

**Directory pattern:**
```
.shipit/
  agents/
    my-custom-agent.json
    researcher.json          ← overrides built-in "researcher"
```

In [17]:
import json
import tempfile
from pathlib import Path

# Create a project-local registry
project_registry = AgentRegistry([custom_agent])

# Override the built-in "researcher" with a custom version
custom_researcher = AgentDefinition(
    id="researcher",  # same ID — will override built-in
    name="Research Specialist (Custom)",
    role="Domain-specific research agent for our fintech product",
    goal="Research financial regulations and compliance requirements",
    prompt="You are a fintech compliance researcher. Focus on SEC, FINRA, and EU regulations.",
    tools=["read_file", "web_search", "open_url"],
    category="Research",
)

custom_reviewer = AgentDefinition(
    id="data-pipeline-reviewer",
    name="Data Pipeline Reviewer (Custom)",
    role="Expert in reviewing data pipelines for reliability and performance",
    goal="Review data pipelines and provide actionable feedback",
    backstory="15 years of experience building and reviewing data pipelines at top tech companies. Deep expertise in Spark, Airflow, dbt, and data quality best practices.",
    model="sonnet",
    tools=["read_file", "grep_files", "glob_files", "bash"],
    max_iterations=10,
    prompt="""Review data pipelines with focus on:
        1. Data quality checks — are there null checks, schema validation, dedup?
        2. Idempotency — can the pipeline be safely re-run?
        3. Error handling — what happens when upstream sources fail?
        4. Performance — are there unnecessary full-table scans?
        5. Monitoring — are there alerts for SLA breaches?
        Output a structured review with findings and recommendations.""",
    category="Data Engineering",
    tags=["data", "pipeline", "etl", "review"],
)

merged = registry.merge(
    AgentRegistry([custom_researcher, custom_reviewer, custom_agent])
)
print(f"Built-in: {len(registry)} agents → Merged: {len(merged)} agents")

r = merged.get("researcher")
print(f"\n'researcher' after merge: {r.name}")
print(f"  Role: {r.role}")

d = merged.get("data-pipeline-reviewer")
print(f"\nCustom agent found: {d.name}")

# Load from directory (simulates .shipit/agents/)
with tempfile.TemporaryDirectory() as tmpdir:
    agent_data = {
        "id": "compliance-checker",
        "name": "Compliance Checker",
        "role": "Regulatory compliance specialist",
        "goal": "Check code for compliance",
        "tools": ["read_file", "grep_files"],
        "prompt": "Review for GDPR and PCI-DSS.",
        "category": "Compliance",
        "tags": ["compliance", "gdpr"],
    }
    with open(Path(tmpdir) / "compliance-checker.json", "w") as f:
        json.dump(agent_data, f)

    local = AgentRegistry.from_directory(tmpdir)
    full = registry.merge(local)
    print(f"\nFrom directory: {len(local)} agent → Total: {len(full)} agents")
    print(
        f"  {full.get('compliance-checker').name}: {full.get('compliance-checker').role}"
    )

Built-in: 40 agents → Merged: 42 agents

'researcher' after merge: Research Specialist (Custom)
  Role: Domain-specific research agent for our fintech product

Custom agent found: Data Pipeline Reviewer (Custom)

From directory: 1 agent → Total: 41 agents
  Compliance Checker: Regulatory compliance specialist


## 8. Using Prebuilt Agents with ShipCrew

Prebuilt agents integrate with `ShipCrew` for multi-agent orchestration. See notebook 33 for full examples.

In [19]:
from shipit_agent.deep.ship_crew import ShipAgent, ShipTask, ShipCrew

researcher_def = registry.get("researcher")
writer_def = registry.get("blog-writer")
reviewer_def = registry.get("code-reviewer")

researcher_sa = ShipAgent(
    name="researcher",
    agent=Agent.with_builtins(llm=llm, prompt=researcher_def.system_prompt()),
    role=researcher_def.role,
    goal=researcher_def.goal,
    backstory=researcher_def.backstory,
)

writer_sa = ShipAgent(
    name="writer",
    agent=Agent.with_builtins(llm=llm, prompt=writer_def.system_prompt()),
    role=writer_def.role,
    goal=writer_def.goal,
    backstory=writer_def.backstory,
)

reviewer_sa = ShipAgent(
    name="reviewer",
    agent=Agent.with_builtins(llm=llm, prompt=reviewer_def.system_prompt()),
    role=reviewer_def.role,
    goal=reviewer_def.goal,
    backstory=reviewer_def.backstory,
)

mini_crew = ShipCrew(
    name="registry-powered-content-crew",
    coordinator_llm=llm,
    agents=[researcher_sa, writer_sa, reviewer_sa],
    tasks=[
        ShipTask(
            name="research",
            description="Research agent observability trends",
            agent="researcher",
            output_key="findings",
        ),
        ShipTask(
            name="draft",
            description="Draft a 2 paragraph post from: {findings}",
            agent="writer",
            depends_on=["research"],
            output_key="draft",
        ),
        ShipTask(
            name="review",
            description="Review this draft and tighten it: {draft}",
            agent="reviewer",
            depends_on=["draft"],
        ),
    ],
)

print(f"Researcher: {researcher_sa.role}")
print(f"Writer:     {writer_sa.role}")
print(f"Reviewer:   {reviewer_sa.role}")
print(
    f"\nMini crew ready with {len(mini_crew.agents)} agents and {len(mini_crew.tasks)} tasks."
)
print(
    "Run `mini_crew.run()` to execute it, or see notebook 33 for the full orchestration walkthrough."
)

Researcher: Research specialist who synthesizes information from multiple sources into actionable insights
Writer:     Technical content writer creating engaging, educational blog posts and articles

Ready for ShipCrew orchestration! See notebook 33.


## Summary

| Feature | API |
|---------|-----|
| Load built-in agents | `AgentRegistry.default()` |
| Get agent by ID | `registry.get("security-auditor")` |
| Search agents | `registry.search("security audit")` |
| Browse by category | `registry.list_by_category("Security")` |
| All categories | `registry.categories()` |
| Custom definitions | `AgentDefinition(id=..., role=..., prompt=...)` |
| Merge registries | `registry.merge(other_registry)` |
| Load from directory | `AgentRegistry.from_directory(".shipit/agents/")` |
| Build live agent | `Agent.with_builtins(llm=llm, prompt=def.system_prompt())` |
| Use with ShipCrew | `ShipAgent(...), ShipTask(...), ShipCrew(...)` |

**40 agents** across **8 categories** — ready to use out of the box.